# Hyperparameter Sweep (Optuna)

This notebook iterates through all 15 MVTec AD categories, performs Optuna trials to find the best hyperparameters, and exports the final mappings for both Keras CAE and PatchCore.

**Note:** Ensure you have the MVTec AD dataset available. The script will automatically locate it at `data/raw/mvtec_ad` from the project root.

In [ ]:
import sys
import os
from pathlib import Path
import json

# Ensure dynamic libraries (CUDA/cuDNN from conda/pixi) are loaded
conda_lib = os.path.join(sys.prefix, "lib")
if os.path.exists(conda_lib) and conda_lib not in os.environ.get("LD_LIBRARY_PATH", ""):
    os.environ["LD_LIBRARY_PATH"] = f"{conda_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"

# Dynamically find the project root (handling both Colab and local VS Code Jupyter)
current_dir = Path.cwd()
while not (current_dir / "app").exists() and current_dir != current_dir.parent:
    current_dir = current_dir.parent
PROJECT_ROOT = current_dir

sys.path.append(str(PROJECT_ROOT))

from app.pipelines.modelling.keras_cae.optuna_study import run_study as run_keras_study
from app.pipelines.modelling.patchcore_optuna_study import run_study as run_patchcore_study

DATA_ROOT = str(PROJECT_ROOT / "data/raw/mvtec_ad")

CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]

keras_results = {}
patchcore_results = {}
for category in CATEGORIES:
    print(f"\n{'='*50}")
    print(f"Running Optuna study for: {category}")
    print(f"{'='*50}")
    
    print("\n--- Keras CAE ---")
    keras_cfg = run_keras_study(category_name=category, n_trials=30, data_root=DATA_ROOT)
    keras_results[category] = keras_cfg
    
    print("\n--- PatchCore ---")
    patchcore_cfg = run_patchcore_study(category_name=category, n_trials=30, data_root=DATA_ROOT)
    patchcore_results[category] = patchcore_cfg

keras_output = Path("keras_cae_best.json")
keras_output.write_text(json.dumps(keras_results, indent=2))

patchcore_output = Path("patchcore_best.json")
patchcore_output.write_text(json.dumps(patchcore_results, indent=2))

print(f"\nSweep complete.")
print(f"Keras CAE hyperparameters saved to {keras_output.resolve()}")
print(f"PatchCore hyperparameters saved to {patchcore_output.resolve()}")